# NCI1: Vanilla vs VN vs B-Cos vs B-Cos+VN

This notebook compares four models on NCI1 with the same evaluation protocol:
- Vanilla GINE
- Vanilla GINE + Virtual Node
- B-Cos GINE
- B-Cos GINE + Virtual Node

Metrics are reported for both:
- Dynamic thresholding (validation-selected threshold)
- Fixed decision rule (argmax)

In [1]:
import sys
from pathlib import Path
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, MessagePassing, global_add_pool
from torch_geometric.nn.aggr import SumAggregation
from bcos.modules import BcosLinear, BcosSequential

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Repo root added: {project_root}')

/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [2]:
dataset = TUDataset(root='data/TUDataset', name='NCI1')
labels = np.array([int(d.y.item()) for d in dataset])

print(f'Dataset: {dataset.name}')
print(f'Graphs: {len(dataset)} | Features: {dataset.num_features} | Classes: {dataset.num_classes}')
print(f'Class 0: {(labels == 0).sum()} | Class 1: {(labels == 1).sum()}')

def get_split_loaders(seed: int, batch_size: int = 128):
    all_idx = np.arange(len(dataset))
    train_idx, temp_idx, y_train, y_temp = train_test_split(
        all_idx, labels, test_size=0.2, stratify=labels, random_state=seed
    )
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=seed
    )

    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    test_ds = dataset[torch.tensor(test_idx)]

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

edge_dim_raw = dataset[0].edge_attr.size(1) if dataset[0].edge_attr is not None else 1
print(f'Raw edge feature dim (fallback to 1 if None): {edge_dim_raw}')

Dataset: NCI1
Graphs: 4110 | Features: 37 | Classes: 2
Class 0: 2053 | Class 1: 2057
Raw edge feature dim (fallback to 1 if None): 1


## Model Definitions

In [3]:
class VanillaGINE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=5, num_classes=2, dropout=0.5):
        super().__init__()
        self.lin_node = nn.Linear(node_dim, hidden_dim)
        self.lin_edge = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
            )
            self.convs.append(GINEConv(nn=mlp, train_eps=True))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.post_conv = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        edge_attr = self.lin_edge(edge_attr)
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_attr)
            x = self.batch_norms[i](x)
            x = F.relu(x)
        x = global_add_pool(x, batch)
        return self.post_conv(x)


class VanillaGINEVirtualNode(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=5, num_classes=2, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.edge_encoder = nn.Linear(edge_dim, hidden_dim)
        self.virtualnode_embedding = nn.Embedding(1, hidden_dim)

        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.vn_mlps = nn.ModuleList()

        for _ in range(num_layers):
            conv_mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
            )
            self.convs.append(GINEConv(nn=conv_mlp, train_eps=True))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            self.vn_mlps.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                    nn.Linear(hidden_dim, hidden_dim),
                )
            )

        self.post_conv = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.node_encoder(x.float())
        edge_attr = self.edge_encoder(edge_attr.float())

        num_graphs = int(batch.max().item()) + 1
        vn_state = self.virtualnode_embedding(
            torch.zeros(num_graphs, dtype=torch.long, device=x.device)
        )

        for layer_idx in range(self.num_layers):
            x = x + vn_state[batch]
            x = self.convs[layer_idx](x, edge_index, edge_attr)
            x = self.batch_norms[layer_idx](x)
            x = F.relu(x)

            if layer_idx < self.num_layers - 1:
                summary = global_add_pool(x, batch)
                vn_state = self.vn_mlps[layer_idx](vn_state + summary)
                vn_state = F.relu(vn_state)

        x = global_add_pool(x, batch)
        return self.post_conv(x)


class BcosGINEConv(MessagePassing):
    def __init__(self, channels, edge_dim, b=2.0, max_out=1, eps=0.0, train_eps=False, **kwargs):
        kwargs.setdefault('aggr', 'add')
        super().__init__(**kwargs)
        self.transform = BcosSequential(
            *[BcosLinear(din, dout, b=b, max_out=max_out) for din, dout in zip(channels[:-1], channels[1:])]
        )
        if train_eps:
            self.eps = nn.Parameter(torch.tensor([eps], dtype=torch.float))
        else:
            self.register_buffer('eps', torch.tensor([eps], dtype=torch.float))

    def forward(self, x, edge_index, edge_attr):
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = (1 + self.eps) * x + out
        return self.transform(out)

    def message(self, x_j, edge_attr):
        return F.relu(x_j + edge_attr)


class PureBcosGINE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=5, num_classes=2, b=2.0, max_out=1, dropout=0.5):
        super().__init__()
        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)
        self.convs = nn.ModuleList([
            BcosGINEConv([hidden_dim, hidden_dim], edge_dim=hidden_dim, b=b, max_out=max_out)
            for _ in range(num_layers)
        ])
        self.readout_mlp = BcosSequential(
            BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out),
        )
        self.dropout_layer = nn.Dropout(dropout)
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        e = self.lin_edge(edge_attr)
        for conv in self.convs:
            x = conv(x, edge_index, e)
        node_logits = self.readout_mlp(x)
        node_logits = self.dropout_layer(node_logits)
        return self.agg(node_logits, batch)


class PureBcosGINEVirtualNode(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=5, num_classes=2, b=2.0, max_out=1, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)
        self.virtualnode_embedding = nn.Embedding(1, hidden_dim)
        self.convs = nn.ModuleList([
            BcosGINEConv([hidden_dim, hidden_dim], edge_dim=hidden_dim, b=b, max_out=max_out)
            for _ in range(num_layers)
        ])
        self.vn_mlps = nn.ModuleList([
            BcosSequential(
                BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
                BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            )
            for _ in range(max(1, num_layers - 1))
        ])
        self.readout_mlp = BcosSequential(
            BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out),
        )
        self.dropout_layer = nn.Dropout(dropout)
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        e = self.lin_edge(edge_attr)
        num_graphs = int(batch.max().item()) + 1
        vn_state = self.virtualnode_embedding(
            torch.zeros(num_graphs, dtype=torch.long, device=x.device)
        )

        for layer_idx, conv in enumerate(self.convs):
            x = x + vn_state[batch]
            x = conv(x, edge_index, e)
            x = F.relu(x)
            if layer_idx < self.num_layers - 1:
                summary = self.agg(x, batch)
                vn_state = self.vn_mlps[layer_idx](vn_state + summary)
                vn_state = F.relu(vn_state)

        node_logits = self.readout_mlp(x)
        node_logits = self.dropout_layer(node_logits)
        return self.agg(node_logits, batch)

In [4]:
def get_edge_attr_or_default(data, edge_dim, device):
    if data.edge_attr is None:
        return torch.ones((data.edge_index.size(1), edge_dim), dtype=torch.float, device=device)
    edge_attr = data.edge_attr.to(device)
    if edge_attr.dim() == 1:
        edge_attr = edge_attr.unsqueeze(-1)
    return edge_attr.float()

def train_one_epoch(model, loader, optimizer, criterion, device, edge_dim):
    model.train()
    total_loss = 0.0
    for data in loader:
        data = data.to(device)
        edge_attr = get_edge_attr_or_default(data, edge_dim, device)
        optimizer.zero_grad()
        out = model(data.x.float(), data.edge_index, edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def collect_scores(model, loader, device, edge_dim):
    model.eval()
    y_true, y_logits = [], []
    for data in loader:
        data = data.to(device)
        edge_attr = get_edge_attr_or_default(data, edge_dim, device)
        out = model(data.x.float(), data.edge_index, edge_attr, data.batch)
        score = out[:, 1] - out[:, 0]
        y_true.extend(data.y.cpu().numpy())
        y_logits.extend(score.cpu().numpy())
    return np.array(y_true), np.array(y_logits)

def compute_optimal_f1(y_true, y_logits):
    y_probs = torch.sigmoid(torch.tensor(y_logits)).numpy()
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]
    if len(f1_scores) == 0:
        return 0.0, 0.0, 0.5
    best_idx = int(np.argmax(f1_scores))
    best_f1 = float(f1_scores[best_idx])
    best_thresh = float(thresholds[best_idx])
    y_pred = (y_probs >= best_thresh).astype(int)
    best_acc = float(accuracy_score(y_true, y_pred))
    return best_f1, best_acc, best_thresh

@torch.no_grad()
def evaluate_with_threshold(model, loader, device, edge_dim, threshold):
    y_true, y_logits = collect_scores(model, loader, device, edge_dim)
    y_probs = torch.sigmoid(torch.tensor(y_logits)).numpy()
    y_pred = (y_probs >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    return float(f1), float(acc)

@torch.no_grad()
def evaluate_argmax(model, loader, device, edge_dim):
    model.eval()
    y_true, y_pred = [], []
    for data in loader:
        data = data.to(device)
        edge_attr = get_edge_attr_or_default(data, edge_dim, device)
        out = model(data.x.float(), data.edge_index, edge_attr, data.batch)
        pred = out.argmax(dim=1)
        y_true.extend(data.y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
    f1 = f1_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    return float(f1), float(acc)

def run_seed_experiment(model_cls, model_name, seeds, hidden_dim=64, num_layers=5, dropout=0.5, lr=1e-3, epochs=80, batch_size=128, b=2.0, max_out=1):
    results = {
        'val_f1': [],
        'test_f1_dynamic': [],
        'test_acc_dynamic': [],
        'test_f1_fixed': [],
        'test_acc_fixed': [],
        'val_thresh': [],
    }
    edge_dim = edge_dim_raw

    print(f"\n--- Running {model_name} ---")
    print('Dynamic = val-selected threshold | Fixed = argmax')

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        train_loader, val_loader, test_loader = get_split_loaders(seed=seed, batch_size=batch_size)

        model = model_cls(
            node_dim=dataset.num_features,
            edge_dim=edge_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=dataset.num_classes,
            dropout=dropout,
            b=b,
            max_out=max_out,
        ).to(device) if model_cls in [PureBcosGINE, PureBcosGINEVirtualNode] else model_cls(
            node_dim=dataset.num_features,
            edge_dim=edge_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=dataset.num_classes,
            dropout=dropout,
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_val_f1 = -1.0
        best_val_thresh = 0.5
        best_state = copy.deepcopy(model.state_dict())

        for _ in range(epochs):
            _ = train_one_epoch(model, train_loader, optimizer, criterion, device, edge_dim)
            val_true, val_logits = collect_scores(model, val_loader, device, edge_dim)
            val_f1, _, val_thresh = compute_optimal_f1(val_true, val_logits)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_val_thresh = val_thresh
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        test_f1_dynamic, test_acc_dynamic = evaluate_with_threshold(
            model, test_loader, device, edge_dim, best_val_thresh
        )
        test_f1_fixed, test_acc_fixed = evaluate_argmax(model, test_loader, device, edge_dim)

        results['val_f1'].append(best_val_f1)
        results['test_f1_dynamic'].append(test_f1_dynamic)
        results['test_acc_dynamic'].append(test_acc_dynamic)
        results['test_f1_fixed'].append(test_f1_fixed)
        results['test_acc_fixed'].append(test_acc_fixed)
        results['val_thresh'].append(best_val_thresh)

        print(
            f"Seed {seed}: Val F1={best_val_f1:.4f} | "
            f"Dynamic Test F1={test_f1_dynamic:.4f}, Acc={test_acc_dynamic:.4f} | "
            f"Fixed Test F1={test_f1_fixed:.4f}, Acc={test_acc_fixed:.4f}"
        )

    return results

In [6]:
SEEDS = [42, 123, 999, 7, 2024]
HIDDEN_DIM = 64
NUM_LAYERS = 5
DROPOUT = 0.5
LR = 1e-3
EPOCHS = 80
BATCH_SIZE = 128

vanilla_results = run_seed_experiment(
    VanillaGINE, 'Vanilla GINE', SEEDS,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE,
 )

vanilla_vn_results = run_seed_experiment(
    VanillaGINEVirtualNode, 'Vanilla GINE + Virtual Node', SEEDS,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE,
 )

bcos_results = run_seed_experiment(
    PureBcosGINE, 'B-Cos GINE', SEEDS,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, b=2.0, max_out=1,
 )

bcos_vn_results = run_seed_experiment(
    PureBcosGINEVirtualNode, 'B-Cos GINE + Virtual Node', SEEDS,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, b=2.0, max_out=1,
 )

def summarize_model(model_name, results):
    dyn_f1_mean = np.mean(results['test_f1_dynamic'])
    dyn_f1_std = np.std(results['test_f1_dynamic'])
    dyn_acc_mean = np.mean(results['test_acc_dynamic'])
    dyn_acc_std = np.std(results['test_acc_dynamic'])
    fix_f1_mean = np.mean(results['test_f1_fixed'])
    fix_f1_std = np.std(results['test_f1_fixed'])
    fix_acc_mean = np.mean(results['test_acc_fixed'])
    fix_acc_std = np.std(results['test_acc_fixed'])
    print(
        f"{model_name:22} | "
        f"Dyn F1: {dyn_f1_mean:.4f}±{dyn_f1_std:.4f}, Dyn Acc: {dyn_acc_mean:.4f}±{dyn_acc_std:.4f} | "
        f"Fix F1: {fix_f1_mean:.4f}±{fix_f1_std:.4f}, Fix Acc: {fix_acc_mean:.4f}±{fix_acc_std:.4f}"
    )

print('\n=== NCI1 Cross-Model Summary (Test Set, Mean±Std) ===')
summarize_model('Vanilla GINE', vanilla_results)
summarize_model('Vanilla GINE + VN', vanilla_vn_results)
summarize_model('B-Cos GINE', bcos_results)
summarize_model('B-Cos GINE + VN', bcos_vn_results)


--- Running Vanilla GINE ---
Dynamic = val-selected threshold | Fixed = argmax
Seed 42: Val F1=0.8456 | Dynamic Test F1=0.7955, Acc=0.7956 | Fixed Test F1=0.7847, Acc=0.7859
Seed 42: Val F1=0.8456 | Dynamic Test F1=0.7955, Acc=0.7956 | Fixed Test F1=0.7847, Acc=0.7859
Seed 123: Val F1=0.8093 | Dynamic Test F1=0.7759, Acc=0.7762 | Fixed Test F1=0.7785, Acc=0.7786
Seed 123: Val F1=0.8093 | Dynamic Test F1=0.7759, Acc=0.7762 | Fixed Test F1=0.7785, Acc=0.7786
Seed 999: Val F1=0.7887 | Dynamic Test F1=0.8029, Acc=0.8029 | Fixed Test F1=0.8029, Acc=0.8029
Seed 999: Val F1=0.7887 | Dynamic Test F1=0.8029, Acc=0.8029 | Fixed Test F1=0.8029, Acc=0.8029
Seed 7: Val F1=0.8188 | Dynamic Test F1=0.7881, Acc=0.7883 | Fixed Test F1=0.7732, Acc=0.7737
Seed 7: Val F1=0.8188 | Dynamic Test F1=0.7881, Acc=0.7883 | Fixed Test F1=0.7732, Acc=0.7737
Seed 2024: Val F1=0.8009 | Dynamic Test F1=0.7617, Acc=0.7640 | Fixed Test F1=0.7806, Acc=0.7810

--- Running Vanilla GINE + Virtual Node ---
Dynamic = val-se